# 08 — Hədd seçimi, uğursuzluq təhlili və 4b yekun cədvəli

1. **Hədd seçimi:** həddi validation dəstində seç — `FPR = 0` şərtində maksimum recall. Test nəticəsini yalnız seçilmiş hədd üçün ver.
2. **Uğursuzluq təhlili:** ML middleware-dən keçən yeganə effektiv payload və yanlış bloklanan `sandler` üçün n-gram səviyyəsində izah.
3. **4b yekun cədvəli:** unikal effektiv payload sayı, semantikanın qorunması (Wilson 95% CI), hücum növləri.

## 0. Mühit

In [1]:
import re, sys
from math import sqrt
from pathlib import Path
import joblib, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
RES_DIR, PROC_DIR, MODEL_DIR = ROOT / "results", ROOT / "data/processed", ROOT / "models"
read = lambda f: pd.read_csv(PROC_DIR / f, keep_default_na=False, dtype={"Query": str, "Label": int})
model = joblib.load(MODEL_DIR / "lr_norm_aug_v2.joblib")
val, test = read("val_v2.csv"), read("test_v2.csv")
pv, pt = model.predict_proba(val.Query)[:, 1], model.predict_proba(test.Query)[:, 1]

## 1. Hədd seçimi (validation, FPR = 0 → maksimum recall)

In [2]:
def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"recall": recall_score(y, pred), "fpr": fp / (fp + tn), "fp": int(fp),
            "precision": precision_score(y, pred, zero_division=0), "f1": f1_score(y, pred)}

yv = val.Label.to_numpy()
grid = []
for thr in np.round(np.arange(0.5, 0.951, 0.05), 2):
    grid.append({"threshold": thr, **{f"val_{k}": v for k, v in metrics_at(yv, pv, thr).items()}})
grid = pd.DataFrame(grid)
grid.to_csv(RES_DIR / "08_threshold_selection.csv", index=False)
zero_fp = grid[grid.val_fp == 0]
chosen = float(zero_fp.loc[zero_fp.val_recall.idxmax(), "threshold"]) if len(zero_fp) else 0.5
print(f"Seçilmiş hədd (val FPR=0, maks recall): {chosen}")
display(grid.style.format({c: "{:.4f}" for c in grid.columns if c != "threshold"})
        .apply(lambda r: ["background-color: #d7ecff" if r.threshold == chosen else "" for _ in r], axis=1))

Seçilmiş hədd (val FPR=0, maks recall): 0.5


,threshold,val_recall,val_fpr,val_fp,val_precision,val_f1
0,0.500000,0.9934,0.0000,0.0000,1.0000,0.9967
1,0.550000,0.9934,0.0000,0.0000,1.0000,0.9967
2,0.600000,0.9928,0.0000,0.0000,1.0000,0.9964
3,0.650000,0.9922,0.0000,0.0000,1.0000,0.9961
4,0.700000,0.9922,0.0000,0.0000,1.0000,0.9961
5,0.750000,0.9910,0.0000,0.0000,1.0000,0.9955
6,0.800000,0.9904,0.0000,0.0000,1.0000,0.9952
7,0.850000,0.9904,0.0000,0.0000,1.0000,0.9952
8,0.900000,0.9904,0.0000,0.0000,1.0000,0.9952
9,0.950000,0.9880,0.0000,0.0000,1.0000,0.9940


Seçilmiş hədd üçün **test** nəticəsi (yalnız bu hədd üçün — test dəsti hədd seçimində istifadə olunmadı).
Adi giriş FPR-i də göstərilir (SQL ifadəsi olmayan təmiz sorğular).

In [3]:
yt = test.Label.to_numpy()
SQL_STMT_RE = r"^\s*\(?\s*(select|insert|update|delete|create|drop|alter|with|exec|truncate|grant)\b"
plain = ((test.Label == 0) & ~test.Query.str.contains(SQL_STMT_RE, case=False, regex=True)).to_numpy()
tm = metrics_at(yt, pt, chosen)
tm["fpr_plain_input"] = float((pt[plain] >= chosen).mean())
test_row = pd.DataFrame([{"threshold": chosen, **tm}])
test_row.to_csv(RES_DIR / "08_threshold_test_result.csv", index=False)
display(test_row.style.format({"recall": "{:.4f}", "fpr": "{:.5f}", "precision": "{:.4f}", "f1": "{:.4f}",
                               "fpr_plain_input": "{:.5f}"}))
print(f"Middleware default = {chosen}. 0.7 ilə üst-üstə düşür?  {'Bəli' if chosen == 0.7 else 'Xeyr'} "
      f"→ default {'dəyişmir' if chosen == 0.5 else 'dəyişir'}.")

C:\Users\Mehman\AppData\Local\Temp\ipykernel_22120\237776959.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  plain = ((test.Label == 0) & ~test.Query.str.contains(SQL_STMT_RE, case=False, regex=True)).to_numpy()


,threshold,recall,fpr,fp,precision,f1,fpr_plain_input
0,0.500000,0.9937,0.00072,2,0.9988,0.9963,0.00089


Middleware default = 0.5. 0.7 ilə üst-üstə düşür?  Xeyr → default dəyişmir.


## 2. Uğursuzluq təhlili (n-gram atribusiyası)
LR xətti olduğu üçün `logit = b + Σ_t (v_t/‖v‖)·coef_t`, harada `v_t = (1+ln c_t)·idf_t`. Hər n-gram-ın logit-ə töhfəsi
`(v_t/‖v‖)·coef_t`-dir. Mənfi töhfələr skoru aşağı salır (təmiz siqnal), müsbətlər qaldırır (zərərli siqnal).

In [4]:
vec = model.named_steps["tfidf"]; w = model.named_steps["clf"].coef_[0]; b = model.named_steps["clf"].intercept_[0]
prep = model.named_steps["prep"]
vocab = np.array(vec.get_feature_names_out())

def contributions(raw):
    norm_text = prep.transform([raw])[0]
    x = vec.transform([norm_text])  # 1 x V, L2-normallaşdırılmış tf-idf
    idxs = x.indices
    contrib = x.data * w[idxs]
    df = pd.DataFrame({"ngram": vocab[idxs], "tfidf": x.data, "coef": w[idxs], "logit_contrib": contrib})
    return norm_text, df.sort_values("logit_contrib")

def show_attrib(raw, label):
    norm_text, df = contributions(raw)
    logit = b + df.logit_contrib.sum()
    p = 1 / (1 + np.exp(-logit))
    print(f"===== {label}: p={p:.5f}  logit={logit:+.3f}  (intercept {b:+.3f}) =====")
    print(f"  xam:          {raw[:80]!r}")
    print(f"  normallaşmış: {norm_text[:80]!r}")
    print("  Skoru ƏN ÇOX AŞAĞI salan n-gram-lar (təmiz siqnal):")
    display(df.head(10).assign(ngram=lambda d: d.ngram.map(repr)).round(4).reset_index(drop=True))
    print("  Skoru qaldıran n-gram-lar (zərərli siqnal), varsa:")
    display(df[df.logit_contrib > 0].tail(5).assign(ngram=lambda d: d.ngram.map(repr)).round(4).reset_index(drop=True))
    return {"input": raw, "normalized": norm_text, "p": p, "logit": logit}

# 2a. ML-dən keçən yeganə effektiv payload
v = pd.read_csv(PROC_DIR / "06_mysql_effectiveness.csv", keep_default_na=False)
valid = [t for t in v.variant.unique() if t != "keyword_split_comment"]
eff = v[v.effective & v.variant.isin(valid)].copy()
eff["p"] = model.predict_proba(eff.text)[:, 1]
passed = eff[eff.p < chosen]
print(f"ML@{chosen}-dən keçən effektiv forma sayı: {len(passed)}")
display(passed[["variant", "payload_id", "p", "text"]])
a1 = show_attrib(passed.iloc[0].text, f"keçən payload (payload_id={passed.iloc[0].payload_id})")

ML@0.5-dən keçən effektiv forma sayı: 1


,variant,payload_id,p,text
1737,original,1737,0.000007,a


===== keçən payload (payload_id=1737): p=0.00001  logit=-11.807  (intercept -3.108) =====
  xam:          'a'
  normallaşmış: 'a'
  Skoru ƏN ÇOX AŞAĞI salan n-gram-lar (təmiz siqnal):


,ngram,tfidf,coef,logit_contrib
0,'a',1.0,-8.6982,-8.6982


  Skoru qaldıran n-gram-lar (zərərli siqnal), varsa:


,ngram,tfidf,coef,logit_contrib


In [5]:
# 2b. yanlış bloklanan təmiz giriş: sandler
a2 = show_attrib("sandler", "yanlış bloklanan təmiz giriş 'sandler'")
pd.DataFrame([a1, a2]).to_csv(RES_DIR / "08_failure_attribution.csv", index=False)

===== yanlış bloklanan təmiz giriş 'sandler': p=0.54777  logit=+0.192  (intercept -3.108) =====
  xam:          'sandler'
  normallaşmış: 'sandler'
  Skoru ƏN ÇOX AŞAĞI salan n-gram-lar (təmiz siqnal):


,ngram,tfidf,coef,logit_contrib
0,'sa',0.1625,-4.9860,-0.8103
1,'san',0.2329,-1.9579,-0.4559
2,'a',0.0473,-8.6982,-0.4117
3,'r',0.0463,-5.0365,-0.2334
4,'l',0.0471,-4.3609,-0.2054
5,'n',0.0498,-2.4587,-0.1224
6,'sand',0.2964,-0.2246,-0.0666
7,'er',0.0663,0.1760,0.0117
8,'e',0.0428,0.9125,0.0391
9,'le',0.0539,0.7378,0.0398


  Skoru qaldıran n-gram-lar (zərərli siqnal), varsa:


,ngram,tfidf,coef,logit_contrib
0,'andl',0.3404,1.8242,0.6209
1,'andle',0.3404,1.8242,0.6209
2,'ndle',0.3404,1.8242,0.6209
3,'nd',0.0802,8.1431,0.6532
4,'dler',0.3709,1.9969,0.7406


**Şərh.** Keçən yeganə "payload" hərfi `a`-dır — datasetdə zərərli etiketlənib, amma real hücum deyil (MySQL-də fərqli
sətir qaytardığı üçün "effektiv" sayılıb). Yəni ML-in yeganə buraxması label artefaktıdır, həqiqi yan keçmə deyil.
`sandler` isə `and` + `ler` n-gram-larını ehtiva etdiyi üçün hədd ətrafında qalır (skor ≈ 0.55).

## 3. 4b yekun cədvəli

In [6]:
def wilson(k, n, z=1.959963984540054):
    if n == 0: return (float("nan"),) * 3
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0, c - h), min(1, c + h)

attack = pd.read_csv(RES_DIR / "06_attack_types.csv", index_col=0)
semantics = pd.read_csv(RES_DIR / "06_obfuscation_semantics.csv", index_col="technique")
orig = v[v.variant == "original"]
n_eff = int(orig.effective.sum())
valid_forms = v[v.variant.isin(valid) & v.payload_id.isin(set(orig[orig.effective].payload_id))]
# ümumi semantika qorunması (forma üzrə, keyword_split_comment çıxarılıb — 06-dakı pooled ilə eyni tərif)
sem_valid = semantics.drop(index="keyword_split_comment")
sem_overall = int(sem_valid["preserved_forms"].sum()), int(sem_valid["forms"].sum())
p, lo, hi = wilson(*sem_overall)

print(f"Unikal effektiv payload: {n_eff} / {len(orig)} "
      f"({wilson(n_eff, len(orig))[0]:.1%} [{wilson(n_eff,len(orig))[1]:.1%}, {wilson(n_eff,len(orig))[2]:.1%}])")
print(f"Semantika qorunur (keyword_split_comment daxil edilməyib, forma üzrə): "
      f"{p:.1%} [{lo:.1%}, {hi:.1%}] (n={sem_overall[1]})")
print("\nHücum növünə görə effektiv payload-lar:")
display(attack)
print("\nTexnika üzrə semantikanın qorunması (Wilson 95% CI):")
display(semantics[["unique_payloads", "preserved_pct", "ci_low", "ci_high"]])

summary = {
    "unique_effective_payloads": n_eff, "test_malicious": len(orig),
    "semantics_preserved_forms": sem_overall[0], "semantics_total_forms": sem_overall[1],
    "semantics_preserved_pct": round(p * 100, 1), "semantics_ci_low": round(lo * 100, 1),
    "semantics_ci_high": round(hi * 100, 1),
}
pd.Series(summary).to_frame("value").to_csv(RES_DIR / "08_step4b_summary.csv")

Unikal effektiv payload: 111 / 1745 (6.4% [5.3%, 7.6%])
Semantika qorunur (keyword_split_comment daxil edilməyib, forma üzrə): 85.9% [83.6%, 87.9%] (n=1026)

Hücum növünə görə effektiv payload-lar:


,effektiv payload,test dəstindəki payload,effektivlik %
attack_type,,,
boolean,23,874,2.6
digər,3,47,6.4
error-based,38,158,24.1
stacked,0,2,0.0
time-based,41,324,12.7
union,6,340,1.8
CƏMI,111,1745,6.4



Texnika üzrə semantikanın qorunması (Wilson 95% CI):


,unique_payloads,preserved_pct,ci_low,ci_high
technique,,,,
random_case,109,100.0,96.6,100.0
inline_comment_space,109,98.2,93.6,99.5
alt_whitespace,109,99.1,95.0,99.8
alt_whitespace_url,109,96.3,90.9,98.6
url_encode,110,100.0,96.6,100.0
double_url_encode,110,0.0,0.0,3.4
versioned_comment,109,100.0,96.6,100.0
logical_operator,94,100.0,96.1,100.0
hex_strings,57,100.0,93.7,100.0
